# LoRA fine-tuning for Gemma 2B

This notebook fine-tunes `google/gemma-2b` with QLoRA on a small instruction dataset using Google Colab. Replace `DATASET_ID` with a Hugging Face dataset containing a `text` column, or run the included sample data.

> **Access requirement:** Accept the Gemma license on the model card and authenticate with a Hugging Face token that can access the model before running the training cells.

In [ ]:
# Install the libraries used by the notebook.
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub

In [ ]:
import os
import torch
from datasets import Dataset, load_dataset
from huggingface_hub import login

# In Colab, add HF_TOKEN under Runtime > Secrets for safer authentication.
# Alternatively, uncomment the next line and paste a token when prompted.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
else:
    print("Set the HF_TOKEN Colab secret before loading Gemma.")

assert torch.cuda.is_available(), "Enable a GPU runtime in Colab: Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name(0))

In [ ]:
# Configuration
MODEL_ID = "google/gemma-2b"
DATASET_ID = None  # Example: "your-username/your-instruction-dataset"
OUTPUT_DIR = "gemma-2b-lora"
MAX_SEQ_LENGTH = 512

# Set this to the name of a text field if your dataset uses a different one.
TEXT_COLUMN = "text"

In [ ]:
# Load a dataset. If DATASET_ID is not set, use a tiny sample so the notebook
# can be smoke-tested without uploading data. Replace this with real data for useful results.
if DATASET_ID:
    dataset = load_dataset(DATASET_ID, split="train")
else:
    examples = [
        "### Instruction:\nExplain why the sky appears blue.\n\n### Response:\nThe sky appears blue because air molecules scatter shorter blue wavelengths of sunlight more strongly than longer red wavelengths.",
        "### Instruction:\nWrite a Python function that adds two numbers.\n\n### Response:\ndef add(a, b):\n    return a + b",
        "### Instruction:\nGive one benefit of unit tests.\n\n### Response:\nUnit tests catch regressions early and make refactoring safer."
    ]
    dataset = Dataset.from_dict({TEXT_COLUMN: examples})

if TEXT_COLUMN not in dataset.column_names:
    raise ValueError(f"Expected a '{TEXT_COLUMN}' column; found {dataset.column_names}")
dataset = dataset.filter(lambda row: isinstance(row[TEXT_COLUMN], str) and bool(row[TEXT_COLUMN].strip()))
print(dataset)
print(dataset[0][TEXT_COLUMN])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field=TEXT_COLUMN,
    max_seq_length=MAX_SEQ_LENGTH,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    fp16=compute_dtype == torch.float16,
    bf16=compute_dtype == torch.bfloat16,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

In [ ]:
# Test the trained adapter.
from peft import PeftModel

model = PeftModel.from_pretrained(model, OUTPUT_DIR)
prompt = "### Instruction:\nExplain what LoRA is.\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    generated = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7)
print(tokenizer.decode(generated[0], skip_special_tokens=True))

## Next steps

- Replace the sample dataset with a sufficiently large, high-quality instruction dataset.
- Adjust `MAX_SEQ_LENGTH`, batch size, learning rate, and number of epochs for your GPU and data.
- The output directory contains the LoRA adapter, not a full copy of the base model. Keep the base model ID and adapter directory together when deploying.